# Meeting 2 — Text-to-image (Colab)

1. **Runtime → Change runtime type → T4 GPU**.
2. W **Secrets** (ikona klucza po lewej) dodaj `GH_TOKEN` — fine-grained token GitHub z dostępem do `synthetic-data`, *Contents: Read and write* — i włącz dostęp dla tego notebooka.
3. Uruchamiaj komórki po kolei. **Co kilka pul uruchom komórkę „Zapis na GitHub”** — po rozłączeniu Colab kasuje dysk.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Klonowanie, instalacja, dane

In [ ]:
import os
from google.colab import userdata

REPO = "Pabl01802/synthetic-data"
BRANCH = "meeting-2"
token = userdata.get("GH_TOKEN")

if not os.path.exists("/content/project"):
    !git clone -b {BRANCH} https://{token}@github.com/{REPO}.git /content/project
%cd /content/project
!git config user.name "Pabl01802"
!git config user.email "58702798+Pabl01802@users.noreply.github.com"
!git log --oneline -3

In [ ]:
!pip install -q -e .
!synth-data

## Prompty

Te same prompty we wszystkich pulach — zmieniamy tylko jedno ustawienie naraz.
Prompty celują w wyglądy, których brakowało w M1: mały kot w pokoju, zbliżenie głowy w ciepłym świetle, zwinięty brązowy kot.

In [ ]:
PROMPTS_LIST = [
    "a photo of an Egyptian Mau cat, silver coat with distinct black spots, green eyes, sitting indoors",
    "close-up photo of the head of an Egyptian Mau cat, spotted coat, warm light",
    "a photo of a bronze Egyptian Mau cat curled up on a sofa, spotted coat",
    "a small Egyptian Mau cat in a living room, full body, spotted silver coat",
]
PROMPTS = " ".join(f'--prompt "{p}"' for p in PROMPTS_LIST)

from IPython.display import Image, display
def sheet(name):
    display(Image(f"pools/{name}/contact-sheet.jpg", width=900))

## Pula bazowa (domyślne ustawienia SD 1.5)

In [ ]:
!synth-generate --name m2-base {PROMPTS}
sheet("m2-base")

In [ ]:
!synth-train --name m2-base --method synthetic --pool m2-base --tag model=sd15

## Pule z jedną zmianą

Przed nimi obejrzyj contact sheet `m2-base` i zanotuj typy błędów (pręgi zamiast cętek, wygląd Bengala, kreskówka, zdeformowane łapy, kilka kotów…).

In [ ]:
!synth-generate --name m2-guidance-3 {PROMPTS} --guidance 3
sheet("m2-guidance-3")
!synth-train --name m2-guidance-3 --method synthetic --pool m2-guidance-3 --tag guidance=3

In [ ]:
!synth-generate --name m2-guidance-12 {PROMPTS} --guidance 12
sheet("m2-guidance-12")
!synth-train --name m2-guidance-12 --method synthetic --pool m2-guidance-12 --tag guidance=12

In [ ]:
!synth-generate --name m2-steps-15 {PROMPTS} --steps 15
sheet("m2-steps-15")
!synth-train --name m2-steps-15 --method synthetic --pool m2-steps-15 --tag steps=15

In [ ]:
# jeden prompt zamiast czterech: czy różnorodność promptów ma znaczenie
P1 = PROMPTS_LIST[0]
!synth-generate --name m2-one-prompt --prompt "{P1}"
sheet("m2-one-prompt")
!synth-train --name m2-one-prompt --method synthetic --pool m2-one-prompt --tag prompts=1

In [ ]:
# SDXL-Turbo: ~7 GB pobierania, 7.7 GB VRAM (mieści się na T4)
!synth-generate --name m2-sdxl-turbo {PROMPTS} --model sdxl-turbo
sheet("m2-sdxl-turbo")
!synth-train --name m2-sdxl-turbo --method synthetic --pool m2-sdxl-turbo --tag model=sdxl-turbo

## Szybkie porównanie na walidacji (z `m1-oversample` jako odniesieniem)

In [ ]:
import glob, json
import pandas as pd

rows = []
for path in sorted(glob.glob("runs/m2-*/results.json")) + ["runs/m1-oversample/results.json"]:
    r = json.load(open(path))
    for s in r["seeds"]:
        rows.append({"run": r["name"], "seed": s["seed"], "mau_recall": s["egyptian_mau_recall"],
                     "macro_f1": s["macro_f1"], "bengal_recall": s["recall"]["Bengal"]})
df = pd.DataFrame(rows)
df.groupby("run")[["mau_recall", "macro_f1", "bengal_recall"]].agg(["mean", "std"]).round(3)

## Zapis na GitHub (obrazy i `model.pt` są w `.gitignore`)

In [ ]:
!git add pools runs
!git commit -m "m2: pools and validation runs" || echo "nothing to commit"
!git pull --rebase
!git push

## Test wybranej puli

**Najpierw** wpisz wybór do `EXPERIMENTS.md` (lokalnie), scommituj i wypchnij. Dopiero potem uruchom poniższe.
`model.pt` są tylko w tej sesji — jeśli sesja padła, przetrenuj wybrany run tą samą komendą (obrazy odtworzysz z tymi samymi ustawieniami i seedem).

In [ ]:
CHOSEN = "m2-base"  # <- zmień na wybraną pulę
!git pull --rebase
!synth-test --run {CHOSEN}
!git add runs
!git commit -m "m2: test {CHOSEN}"
!git push

## (Opcjonalnie) kopia `model.pt` na Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/synth-m2
!rsync -a --include="*/" --include="model.pt" --exclude="*" runs/ /content/drive/MyDrive/synth-m2/runs/